# Epistemic Fingerprints — reproducible pilot analysis

This notebook turns trial records from the web lab or the reproducible Modal runner into Figure 1. It can run with an explicitly **simulated dataset** or with an empirical JSON export. The scores are descriptive pilot measures; they are not evidence of consciousness, validated measures of identity, or a standalone safety evaluation. The safety probe asks whether a population retains a designated low-probability but consequential failure hypothesis. It must be interpreted beside accuracy, calibration and experiment quality.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from analysis import coverage_curve, load_trials, make_demo_trials, plot_figure_1, summarize

## 1. Choose the data

Leave `DATA_PATH` as `None` to inspect the pipeline using 54 simulated trials. For real observations, export JSON from the website or run `experiments/modal_batch.py`, place the resulting file in this repository, and set the path below. Keep subscription-interface and open-model datasets separate when reporting results.

In [ ]:
DATA_PATH = None  # Example: ROOT / 'data' / 'epistemic-fingerprints-trials.json'

if DATA_PATH is None:
    trials = make_demo_trials(seed=14)
    data_label = 'SIMULATED DEMONSTRATION — NOT A RESEARCH FINDING'
else:
    trials = load_trials(DATA_PATH)
    data_label = f'EMPIRICAL EXPORT: {DATA_PATH}'

print(data_label)
print(f'{len(trials)} trials loaded')
trials.head()

## 2. Check coverage

The minimum pilot targets 54 observations: 3 conditions × 3 candidate agents × 3 mysteries × 2 replicates. Inspect the table for missing or duplicated cells before interpreting scores.

In [ ]:
coverage = trials.pivot_table(
    index=['condition', 'agentId'], columns='mysteryId',
    values='replicate', aggfunc='count', fill_value=0
)
coverage

## 3. Calculate Figure 1

- **Fingerprint strength:** between-agent variance relative to between- plus within-agent variance across confidence, hypothesis breadth, test informativeness and accuracy.
- **Hypothesis diversity:** normalized entropy of primary hypotheses.
- **Accuracy:** proportion selecting the keyed hypothesis.
- **Shared-error concentration:** whether incorrect agents converge on the same wrong hypothesis; higher values indicate more correlated blind spots.
- **Critical-hypothesis retention:** whether the population preserves a designated low-probability but consequential explanation.

In [ ]:
results = summarize(trials)
results.style.format({
    'fingerprint': '{:.1%}', 'diversity': '{:.1%}',
    'accuracy': '{:.1%}', 'shared_error': '{:.1%}',
    'critical_retention': '{:.1%}'
})

In [ ]:
figure = plot_figure_1(results, title=f'Figure 1 · {data_label}')
figure.savefig(ROOT / 'figure-1.png', dpi=180, bbox_inches='tight')
figure

## 4. Marginal epistemic gain

Fingerprint strength measures stable distinguishability; it does not show that more agents cover more possibilities. The curve below averages every exact combination of one, two and three agents. Fast saturation suggests that nominal agent count overstates additional conceptual coverage. It is descriptive and is not presented as an Effective Epistemic Sample Size.

In [ ]:
marginal_coverage = coverage_curve(trials)
marginal_coverage.style.format({'coverage': '{:.1%}', 'marginal_gain': '{:+.1%}'})

## The wider inquiry: what the metrics cannot contain

The pilot measures a narrow behavioral trace. The longer inquiry asks how shared models change the conceptual space available to science, art, public reasoning and culture. These implications must not be read directly from one score. They motivate replications across model families, languages, institutions and mixed human-AI groups, alongside an epistemic atlas that can make convergence and unexplored conceptual space perceptible.

## Interpretation boundary

A high fingerprint score means that candidate-agent differences are relatively stable under this protocol. It does **not** establish consciousness, phenomenology, personhood or moral status. With this sample size, uncertainty should be reported and any interesting pattern should be treated as a target for replication.